In [ ]:
from huggingface_hub import login
import os
import sys
import csv
from tqdm import trange
from transformers import AutoModel,AutoTokenizer
# FILE_PATH = './QA_results_GT.csv'
# os.environ["OPENAI_API_KEY"] = AAA
from tqdm import tqdm
import pandas as pd
import ast
tqdm.pandas()

In [ ]:
# ANA_FILE_PATH = './mthp_output.csv'

# naiveanswer_LIST = []
# lightraganswer_LIST = []
# minianswer_LIST = []
# QUESTION_LIST = []
# GA_LIST = []
# filelength = 0
# with open(ANA_FILE_PATH, mode='r', encoding='utf-8') as question_file:
#     reader = csv.DictReader(question_file)
#     for row in reader:
#         QUESTION_LIST.append(row['Question'])
#         GA_LIST.append(row['Gold Answer'])
#         naiveanswer_LIST.append(row['naive'])
#         lightraganswer_LIST.append(row['lightrag'])
#         minianswer_LIST.append(row['minirag'])
#         filelength = filelength+1

In [ ]:
PROMPT = """
Now, I'll give you a question, a gold answer to this question, and three answers provided by different students.

Determine the answer according to the following rules:
If the answer is correct, get 1 point.
If the answer is irrelevant to the question, it will receive 0 points.
If the answer is incorrect, get -1 point.

Return your answer in JSON mode.

For example:

Question:
When does Li Hua arrive to the city?

Gold Answer:
20260105

Answer1: LiHua arrived on the afternoon of January 5th
Answer2: Sorry, there is no information about LiHua's arrival in the information you provided
Answer3: There is no accurate answer in the information you provided, but according to the first information found, LiHua arrived on April 17th

output:
{{
"Score1": 1,
"Score2": 0,
"Score3": -1,
}}



Real data:

Question:
{question}
Gold Answer:
{ga}

Answer1: {naive}
Answer2: {light}
Answer3: {mini}

output:

"""

In [ ]:
PROMPT="""
You are an evaluator designed to score an answer provided by a student against a gold standard.

Determine the score according to the following rules:
If the Answer is correct, assign 1 point.
If the Answer is irrelevant to the question, assign 0 points.
If the Answer is incorrect, assign -1 point.

The score must be returned in JSON format.

---
Example 1:

Question:
When does Li Hua arrive to the city?

Gold Answer:
20260105

Answer: LiHua arrived on the afternoon of January 5th

output:
{{
"Score": 1
}}

---
Example 2:

Question:
What is Li Hua's favorite day to go to the gym?

Gold Answer:
There is no information about Li Hua's favorite day to go to the gym

Answer: Sorry, there is no information about Li Hua's favorite day to go to the gym

output:
{{
"Score": 1
}}

---
Example 3:

Question:
When does Li Hua reschedule her training session to Friday?

Gold Answer:
20260211

Answer: Li Hua rescheduled her training session to Friday, April 17th.

output:
{{
"Score": -1
}}

---
Real data to evaluate:

Question:
{question}

Gold Answer:
{ga}

Answer: {student_answer}

output:
"""

In [ ]:
import requests
import json

def get_eval_deepseek(ques, gold_answer, response):

    p = PROMPT.format(question = ques, ga = gold_answer,  student_answer = response)
    response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
    "Authorization": "",
    "HTTP-Referer": "", # Optional. Site URL for rankings on openrouter.ai.
    "X-Title": "", # Optional. Site title for rankings on openrouter.ai.
  },
    data=json.dumps({
        "model": "deepseek/deepseek-chat",
        "messages": [
        {
                    "role": "system",
                    "content":p,
        }
        ]
    })
    )
    try:
        return response.json()['choices'][0]['message']['content']
    except:
        print("ERROR")
        return response

In [ ]:
# df=pd.read_csv("/content/relevant_context3 (1).csv")
df=pd.read_csv("/content/relevant_context_v4_llama.csv")

In [ ]:
df.head()

,Question,Gold Answer,query,context_json_data,sys_prompt,minirag,relevant_context,relevant_sys_prompt,relevant_context_response
0,Did Li Hua send a message to Jennifer thanking...,Yes,Did Li Hua send a message to Jennifer thanking...,"[{""id"":0,""content"":""Time: 20260119_14:30\nJenn...",---Role---\n\nYou are a helpful assistant resp...,Based on the information provided in the messa...,Li Hua sent a message to Jennifer thanking her...,---Role---\n\nYou are a helpful assistant resp...,**Question Analysis**\nLi Hua's message to Jen...
1,Did Yuriko ask Li Hua for help with her studio...,Yes,Did Yuriko ask Li Hua for help with her studio...,"[{""id"":0,""content"":""Time: 20260309_10:00\nLiHu...",---Role---\n\nYou are a helpful assistant resp...,"Yes, according to the conversation, Yuriko did...",Yuriko did ask Li Hua for help with her studio...,---Role---\n\nYou are a helpful assistant resp...,### Response Summary\n\nBased on the provided ...
2,Did Li Hua send a message to Jennifer asking i...,Yes,Did Li Hua send a message to Jennifer asking i...,"[{""id"":0,""content"":""Time: 20260309_12:00\nLiHu...",---Role---\n\nYou are a helpful assistant resp...,"Yes, Li Hua did request a change in his traini...",Li Hua sent a message to Jennifer asking if he...,---Role---\n\nYou are a helpful assistant resp...,### Response to User Question\n\nAccording to ...
3,"What time does Li Hua watch the movie ""Overwat...",20260122,"What time does Li Hua watch the movie ""Overwat...","[{""id"":0,""content"":""Time: 20260121_13:00\nWolf...",---Role---\n\nYou are a helpful assistant resp...,"Based on the conversation shared, WolfgangSchu...","Li Hua plans to watch the movie ""Overwatch 3"" ...",---Role---\n\nYou are a helpful assistant resp...,### Movie Time for Li Hua\n\nAccording to the ...
4,"Who does Li Hua go to watch the movie ""Overwat...",Wolfgang,"Who does Li Hua go to watch the movie ""Overwat...","[{""id"":0,""content"":""Time: 20260121_13:00\nWolf...",---Role---\n\nYou are a helpful assistant resp...,Based on the information provided in the conve...,"Li Hua plans to watch the movie ""Overwatch 3"" ...",---Role---\n\nYou are a helpful assistant resp...,**Movie Night Companion**\nLi Hua plans to wat...


In [ ]:
df['relevant_context_response']

,relevant_context_response
0,**Question Analysis**\nLi Hua's message to Jen...
1,### Response Summary\n\nBased on the provided ...
2,### Response to User Question\n\nAccording to ...
3,### Movie Time for Li Hua\n\nAccording to the ...
4,**Movie Night Companion**\nLi Hua plans to wat...
5,**Lunar New Year Wishes for Li Hua**\n========...
6,I couldn't find any specific information on wh...
7,I'm not aware of any information about Wolfgan...
8,### Content of the First-Ever Delivery\n\nAcco...
9,**Analysis of the Conversation**\n\nIn the con...


In [ ]:
a=get_eval_deepseek(df['Question'][0], df['Gold Answer'][0], df['minirag'][0])

In [ ]:
a

'```json\n{\n"Score": -1\n}\n```'

In [ ]:
# df['minirag_eval']=df.apply(lambda x: get_eval_deepseek(x['Question'], x['Gold Answer'], x['minirag']), axis=1)



In [ ]:
df['relevant_minirag_eval']=df.progress_apply(lambda x: get_eval_deepseek(x['Question'], x['Gold Answer'], x['relevant_context_response']), axis=1)



100%|██████████| 58/58 [01:45<00:00,  1.82s/it]


In [ ]:
# df['relevant_minirag_eval'][17]="""```json\n{\n  "Score": 1\n}\n```"""

In [ ]:
df['relevant_minirag_eval'][17]

'```json\n{\n  "Score": 1\n}\n```'

In [ ]:
df['relevant_minirag_acc']=df['relevant_minirag_eval'].progress_apply(lambda x: ast.literal_eval(x.strip("```json").strip("```"))['Score'])

100%|██████████| 58/58 [00:00<00:00, 10115.58it/s]


In [ ]:
# df['minirag_acc']=df['minirag_eval'].apply(lambda x: ast.literal_eval(x.strip("```json").strip("```"))['Score'])

In [ ]:
# df['minirag_acc'].value_counts()

In [ ]:
df['relevant_minirag_acc'].value_counts()

,count
relevant_minirag_acc,
1,36
-1,15
0,7


In [ ]:
df['relevant_minirag_acc'].value_counts()

,count
relevant_minirag_acc,
1,38
-1,14
0,6


In [ ]:
df['relevant_minirag_acc'].value_counts()

,count
relevant_minirag_acc,
1,35
-1,18
0,5


In [ ]:
35/58

0.603448275862069

In [ ]:
# #deepseek
# from openai import OpenAI
# chatbot = OpenAI(api_key=My_deepseek_key, base_url="https://api.deepseek.com")

# chat_list = []
# for i in range(filelength):
#     p = PROMPT.format(question = QUESTION_LIST[i], ga = GA_LIST[i], naive = naiveanswer_LIST[i], light = lightraganswer_LIST[i], mini = minianswer_LIST[i])
#     chat_completion = chatbot.chat.completions.create(
#         messages=[
#             {
#                 "role": "system",
#                 "content":p,
#             },


#         ],
#         model="deepseek-chat",
#         stream = False
#     )
#     chat_list.append(chat_completion.choices[0].message.content.strip())

In [ ]:
# #openai
# from openai import OpenAI
# from tqdm import trange
# chatbot = OpenAI()
# chat_list = []
# for i in trange(filelength):
#     p = PROMPT.format(question = QUESTION_LIST[i], ga = GA_LIST[i], naive = naiveanswer_LIST[i], light = lightraganswer_LIST[i], mini = minianswer_LIST[i])
#     chat_completion = chatbot.chat.completions.create(
#         messages=[
#             {
#                 "role": "system",
#                 "content":p,
#             },
#         ],
#         model="gpt-4o",
#     )
#     chat_list.append(chat_completion.choices[0].message.content.strip())
